# Goal

Проверка гипотезы: построчное управление consistency_loss в зависимости от изменчивости сцены позволит агенту избегать чрезмерной стереотипности экшн планов.

Используем дефолтные параметры `consistency_loss.gate`, но выставляем `consistency_loss.gate.ent_coef=0`.

# Control plane

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_11.ipynb'
OPTUNA_DECISIVE_METRIC = 'rollout/episode_lp_conv'

# @launchit.collect
GRID_SEARCH_SPACE = dict(
    random_seed=list(range(1, 121, 10)),
)
# @launchit.stop

if 'GRID_SEARCH_SPACE' in globals():
    import math
    LAUNCHES_COUNT = math.prod(map(len, GRID_SEARCH_SPACE.values()))
    assert LAUNCHES_COUNT > 0
else:
    LAUNCHES_COUNT = 12

# Manual override
# LAUNCHES_COUNT = 10 * 3 - 13

print(f'{LAUNCHES_COUNT=}')

LAUNCHES_COUNT=12


# set_hyperparameters

In [2]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = optuna_trial.suggest_categorical('random_seed', GRID_SEARCH_SPACE['random_seed'])
    HP.general.is_torch_deterministic = False # нафиг детерминизм
    HP.general.is_torch_compile = True 
    HP.general.is_torch_amp = True  # без этого памяти будет жраться х1,5 и дольше, поэтому must-have
    HP.general.is_torch_amp_on_rollout = False  # AMP на роллауте тоже нафиг
    
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    wm_parent = '18d_world_model_09:224'
    HP.vision_head.parent = dict(model=wm_parent, weights=wm_parent)
    HP.vision_head.is_trainable = False
    HP.vision_head.is_saving = False

    parent = '18n_ppo_tr_frostbite_09:1152'
    HP.encoder.parent = dict(model=wm_parent, weights=parent)
    HP.encoder.is_trainable = True
    HP.vision_head.is_saving = True

    HP.agent.parent = dict(weights=parent, lora=None)
    HP.agent.sequence_length = 33
    HP.agent.sequence_ages = [0, 1, 2, 3, 8, 16, 24, 32] # two time scales with steps 1 (~0.13 second) and 8 (~1 second)
    HP.agent.action_plan_length = 10 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, dim_feedforward=1024, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.lora = None
    HP.agent.is_trainable = True 
    HP.agent.is_saving = dict(weights=True, lora=False)
    
    # Test params
    HP.test.break_on_level_passed = False
    HP.test.envs_count = 30
    HP.test.env_stories = None # just standard game

    # Rollout params
    HP.rollout.steps_count = 512 # how many steps to run in a single policy rolllout
    l5 = lambda cls: f'com.develorium.neurolab.frostbite_ram:level5:1:cls={cls}'
    patch = 'three_lives+no_score+no_igloo+temperature_45+bailey_safe_random_spawn+bear_to_the_right_of_igloo'
    HP.rollout.env_stories = {
        # Main block
        '0:6':   (l5('train1'), patch), 
        '6:12':  (l5('train2'), patch), 
        '12:18': (l5('train3'), patch), 
        '18:24': (l5('train4'), patch), 
        '24:30': (l5('train5'), patch), 
    }
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 256
    HP.ppo.learn_rate = f'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef=f'linear(0.05, 0.02)') 
    HP.ppo.consistency_loss = dict(
        type='MSE', coef=0.1,
        gate=dict(enabled=True, dphi_half=2.0, min_cons=0.35, ent_idle_gain=1.0, ent_coef='const(0)'),
        flat=None,
    )
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization

    # Reporting
    HP.report.rollout_env_groups = None
    HP.report.follow_up_freq = 1

    return HP

# Results

Плюс-минус результаты, как в `52`.

|что|`18n_study_54`|`18n_study_52`|
|---|---|---|
|episode_lp|<img src="./img/episode_lp_54.png">|<img src="./img/episode_lp_52.png">|
|episode_r|<img src="./img/episode_r_54.png">|<img src="./img/episode_r_52.png">|

**Выводы**
1) надо попробовать теперь ещё со штрафом за плоскостопие плана

Первые результаты `18n_study_54`. Конфиг: построчный гейт `consistency_loss` по изменчивости сцены, но с `consistency_loss.gate.ent_coef=0` — чистый гейт, без доп. энтропийного бонуса. Всё остальное как в `18n_study_52` (тот же ноутбук/диета/родитель `09:1152`): one-factor — только гейт отличается от контроля.

Контроль: `18n_study_52`, раны с тем же родителем `18n_ppo_tr_frostbite_09:1152` (n=12), без гейта.

**Итоговые значения @3M (mean ± std по ранам, Welch):**

|метрика|`18n_study_54` (gate, n=6)|`18n_study_52` (ctrl, n=12)|Welch p|
|---|---|---|---|
|`rollout/episode_lp_conv` (решающая)|22.94 ± 5.39|23.72 ± 4.82|**0.770**|
|`rollout/episode_lp_max_conv`|47.95 ± 2.65|47.79 ± 3.84|0.920|
|`rollout/episode_r`|2374 ± 591|2653 ± 600|0.371|
|`losses/action_plan_consistency_loss`|**0.030 ± 0.006**|0.042 ± 0.008|**0.004**|
|`losses/entropy_loss`|1.476 ± 0.095|1.447 ± 0.064|0.524|
|`test/levels_passed_mean`|0.072 ± 0.039|0.158 ± 0.107|0.026|
|`test/returns_mean`|283 ± 40|392 ± 175|0.060|

Пер-рановые значения решающей `rollout/episode_lp_conv` (54): 13.61 / 19.70 / 23.71 / 26.51 / 26.61 / 27.48; (52): 12.14 … 30.45.


**Выводы**
1) Гейт работает механически: остаток `action_plan_consistency_loss` падает 0.042 → 0.030 (p=0.004) — пер-строчные веса реально перераспределяют давление с событийных шагов на скучные.
2) На решающей метрике — **НУЛЬ**: `episode_lp_conv` 22.94 vs 23.72 (p=0.77), `episode_lp_max_conv` 47.95 vs 47.79 (p=0.92). Гейт НЕ вредит, но и НЕ помогает.
3) Перераспределение строгости consistency само по себе не создаёт exploration — подтверждение, что «стереотипность» держится не распределением веса лосса по шагам.
4) `test/levels_passed_mean` чуть ниже у 54 (0.072 vs 0.158, p=0.026), но обе величины — почти пол диеты level5; на решающей метрике не отражается, на выводы не влияет.
5) Следующий шаг — атаковать сам плоский аттрактор плана (штраф за плоскость, `18n_study_55`).

# System

In [3]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [4]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_11.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_11',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_54/18n_study_54.ipynb',
 'optuna_study_name': '18n_study_54',
 'optuna_study_serial': '54',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_54/18n_study_54.optuna'}

In [5]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [6]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [7]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [8]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [9]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while LAUNCHES_COUNT is None or completed_launches_count < LAUNCHES_COUNT:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if LAUNCHES_COUNT is None or (completed_launches_count + len(futures) < LAUNCHES_COUNT):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-11-05 21:04:34,359] A new study created in Journal with name: 18n_study_54


2026.11.05-21:04:34.883872     0.495 >> Model instance registered, version=276
2026.11.05-21:04:34.902468     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_11-launch276.ipynb"
2026.11.05-21:04:34.902924     0.007 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_11:276"; running launches=1
2026.11.05-21:05:06.025473     0.438 >> Model instance registered, version=277
2026.11.05-21:05:06.041255     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_11-launch277.ipynb"
2026.11.05-21:05:06.041582     0.007 >> 5.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_11:277"; running launches=2
2026.11.05-21:05:37.358404     0.521 >> Model instance registered, version=278
2026.11.05-21:05:37.376544     0.007 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_11-launch278.ipynb"
2026.11.05-21:05:37.376813     0.007 >> 4.5 idle runners exist, submitted launch "18n_ppo_tr_frostbite_11

In [10]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-11-05 22:22:13,812] Using an existing study with name '18n_study_54' instead of creating a new one.


2026.11.05-22:22:13.815028     5.038 >> Study statistics: 
2026.11.05-22:22:13.820279     0.005 >> 	Number of finished trials: 12
2026.11.05-22:22:13.821905     0.002 >> 	Number of pruned trials: 0
2026.11.05-22:22:13.822774     0.001 >> 	Number of complete trials: 6
2026.11.05-22:22:13.824164     0.001 >> Best trial:
2026.11.05-22:22:13.825911     0.002 >> 	Value: 27.48496796800994
2026.11.05-22:22:13.826583     0.001 >> 	Model version: 280
2026.11.05-22:22:13.827157     0.001 >> 	Params: 
2026.11.05-22:22:13.827778     0.001 >> 		random_seed: 21


In [11]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.11.05-22:22:13.835116     0.007 >> Top trials:
2026.11.05-22:22:13.836117     0.001 >> 1) 18n_ppo_tr_frostbite_11:280, value=27.48496796800994
2026.11.05-22:22:13.836642     0.001 >> 2) 18n_ppo_tr_frostbite_11:279, value=26.608772643545077
2026.11.05-22:22:13.837033     0.000 >> 3) 18n_ppo_tr_frostbite_11:277, value=26.51045342314509


In [12]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(22.93723621539866), np.float64(4.915373445332394))